# Level 4: Quantum Protocols
## Q# Edition — Superdense Coding & Teleportation

In this notebook, you will:

1. Implement **Quantum Teleportation** with modular Q# operations
2. Implement **Superdense Coding** — send 2 classical bits using 1 qubit
3. Verify correctness with statistics

In [ ]:
import qsharp
from qsharp_widgets import Circuit
print("Q# ready!")

---
## 1. Quantum Teleportation

We'll build teleportation step by step with modular operations.

In [ ]:
%%qsharp

/// Alice's part: entangle her qubit with the Bell pair, then measure
operation AliceSends(msg : Qubit, here : Qubit) : (Result, Result) {
    CNOT(msg, here);
    H(msg);
    return (M(msg), M(here));
}

/// Bob's part: apply corrections based on Alice's measurement results
operation BobReceives(alice_results : (Result, Result), there : Qubit) : Unit {
    let (r1, r2) = alice_results;
    if r2 == One {
        X(there);
    }
    if r1 == One {
        Z(there);
    }
}

/// Full teleportation protocol
operation Teleport(sendOne : Bool) : Result {
    use (msg, here, there) = (Qubit(), Qubit(), Qubit());

    // Prepare the state to teleport
    if sendOne {
        X(msg);
    }

    // Create Bell pair
    H(here);
    CNOT(here, there);

    // Alice sends, Bob receives
    let results = AliceSends(msg, here);
    BobReceives(results, there);

    // Measure Bob's qubit
    let bobResult = M(there);

    ResetAll([msg, here, there]);
    return bobResult;
}

In [ ]:
%%qsharp

operation TeleportMany() : Unit {
    let shots = 1000;

    // Teleport |0⟩
    mutable ones_for_zero = 0;
    for _ in 0..shots - 1 {
        if Teleport(false) == One {
            set ones_for_zero += 1;
        }
    }
    Message($"Teleport |0⟩: Got One {ones_for_zero}/{shots} times (should be ~0)");

    // Teleport |1⟩
    mutable ones_for_one = 0;
    for _ in 0..shots - 1 {
        if Teleport(true) == One {
            set ones_for_one += 1;
        }
    }
    Message($"Teleport |1⟩: Got One {ones_for_one}/{shots} times (should be ~{shots})");
}

TeleportMany()

In [ ]:
# Visualize the teleportation circuit
circuit = qsharp.circuit("Teleport(true)")
Circuit(circuit)

---
## 2. Superdense Coding

Alice can send 2 classical bits to Bob by sending just 1 qubit (plus a pre-shared Bell pair).

In [ ]:
%%qsharp

/// Alice encodes 2 bits onto her qubit
operation AliceEncodes(bit1 : Bool, bit2 : Bool, q : Qubit) : Unit {
    if bit2 {
        X(q);
    }
    if bit1 {
        Z(q);
    }
}

/// Bob decodes by measuring in Bell basis
operation BobDecodes(alice_q : Qubit, bob_q : Qubit) : (Result, Result) {
    CNOT(alice_q, bob_q);
    H(alice_q);
    return (M(alice_q), M(bob_q));
}

/// Full superdense coding protocol
operation SuperdenseCoding(bit1 : Bool, bit2 : Bool) : (Result, Result) {
    use (alice_q, bob_q) = (Qubit(), Qubit());

    // Create Bell pair
    H(alice_q);
    CNOT(alice_q, bob_q);

    // Alice encodes and sends
    AliceEncodes(bit1, bit2, alice_q);

    // Bob decodes
    let result = BobDecodes(alice_q, bob_q);

    ResetAll([alice_q, bob_q]);
    return result;
}

In [ ]:
%%qsharp

operation TestSuperdenseCoding() : Unit {
    let messages = [(false, false), (false, true), (true, false), (true, true)];

    for (b1, b2) in messages {
        mutable correct = 0;
        let shots = 100;

        for _ in 0..shots - 1 {
            let (r1, r2) = SuperdenseCoding(b1, b2);
            let got1 = r1 == One;
            let got2 = r2 == One;
            if got1 == b1 and got2 == b2 {
                set correct += 1;
            }
        }

        let s1 = b1 ? "1" | "0";
        let s2 = b2 ? "1" | "0";
        Message($"Message ({s1},{s2}): {correct}/{shots} correct");
    }
}

TestSuperdenseCoding()

---
## 3. Protocol Comparison

| Protocol | Sends | Uses | Classical Bits | Qubits Sent |
|----------|-------|------|----------------|-------------|
| **Superdense Coding** | 2 classical bits | Bell pair | 0 | 1 |
| **Teleportation** | 1 quantum state | Bell pair | 2 | 0 |

They are **dual** to each other!

---
## 4. Key Takeaways

| Concept | Q# Feature |
|---------|------------|
| **Modular operations** | `AliceSends`, `BobReceives` as separate operations |
| **Tuple returns** | `(Result, Result)` for returning multiple values |
| **Tuple destructure** | `let (r1, r2) = func();` |
| **Ternary** | `condition ? valueTrue \| valueFalse` |
| **ResetAll** | Clean up multiple qubits at once |

---
## 5. Challenges

1. **Teleport superposition**: Modify `Teleport` to apply `H` to the message qubit before teleporting. Verify Bob gets 50/50 results.
2. **Teleport with rotation**: Apply `Ry(1.0)` to the message qubit. Check Bob's statistics match a direct `Ry(1.0)` measurement.
3. **Circuit visualization**: Use `qsharp.circuit()` to visualize the superdense coding circuit.

In [ ]:
%%qsharp

// Your challenge code here!

---
**Next up: [Level 5 — Quantum Oracles & Deutsch-Jozsa](../Level_05_Quantum_Oracles/Level_05_QSharp.ipynb)**

# Level 4: Quantum Protocols — Teleportation & Superdense Coding
## Q# Edition

In this notebook, you will:

1. Understand the **No-Cloning Theorem**
2. Implement **Quantum Teleportation** with modular Alice/Bob operations
3. Implement **Superdense Coding**
4. Leverage Q#'s clean syntax for quantum protocols

In [ ]:
import qsharp
from qsharp_widgets import Circuit
print("Q# ready!")

---
## 1. The No-Cloning Theorem

> **It is impossible to create an exact copy of an arbitrary unknown quantum state.**

But we CAN **teleport** a state: destroy it in one place and recreate it in another, using entanglement and classical communication.

---
## 2. Quantum Teleportation in Q#

Q# excels at modular quantum protocols. We'll implement teleportation as separate operations for Alice and Bob.

### The Protocol:
1. Alice and Bob share a Bell pair
2. Alice entangles her message qubit with her half of the pair
3. Alice measures and sends classical bits to Bob
4. Bob corrects his qubit based on Alice's message

In [ ]:
%%qsharp

// Alice's part: entangle message with her Bell qubit, then measure
operation AliceSends(message : Qubit, aliceBell : Qubit) : (Result, Result) {
    // Entangle message with Alice's Bell qubit
    CNOT(message, aliceBell);
    H(message);

    // Measure both qubits
    let m1 = M(message);
    let m2 = M(aliceBell);

    return (m1, m2);
}

// Bob's part: correct his qubit based on Alice's classical bits
operation BobReceives(bobBell : Qubit, bit1 : Result, bit2 : Result) : Unit {
    if bit2 == One {
        X(bobBell);
    }
    if bit1 == One {
        Z(bobBell);
    }
}

// Full teleportation protocol
operation Teleport(sendOne : Bool) : Result {
    use (message, aliceBell, bobBell) = (Qubit(), Qubit(), Qubit());

    // Prepare the state to teleport
    if sendOne {
        X(message);  // Teleport |1> instead of |0>
    }

    Message($"Teleporting state: |{sendOne ? "1" | "0"}>");

    // Create Bell pair between Alice and Bob
    H(aliceBell);
    CNOT(aliceBell, bobBell);

    // Alice performs her operations and measures
    let (bit1, bit2) = AliceSends(message, aliceBell);
    Message($"Alice's measurements: ({bit1}, {bit2})");

    // Bob corrects based on Alice's classical message
    BobReceives(bobBell, bit1, bit2);

    // Verify: measure Bob's qubit
    let result = M(bobBell);
    Message($"Bob's qubit after teleportation: {result}");

    ResetAll([message, aliceBell, bobBell]);
    return result;
}

In [ ]:
# Teleport |0>
print("=== Teleporting |0> ===")
result = qsharp.eval("Teleport(false)")
print(f"\nFinal result: {result}\n")

# Teleport |1>
print("=== Teleporting |1> ===")
result = qsharp.eval("Teleport(true)")
print(f"\nFinal result: {result}")

In [ ]:
# Visualize the teleportation circuit
Circuit(qsharp.circuit("Teleport(true)"))

### Verifying Teleportation Statistically

Let's run it many times to confirm the state is always correctly teleported:

In [ ]:
%%qsharp

operation TeleportMany(sendOne : Bool, shots : Int) : (Int, Int) {
    mutable zeros = 0;
    mutable ones = 0;

    for _ in 0..shots - 1 {
        use (msg, a, b) = (Qubit(), Qubit(), Qubit());

        if sendOne { X(msg); }

        // Bell pair
        H(a);
        CNOT(a, b);

        // Alice
        let (bit1, bit2) = AliceSends(msg, a);

        // Bob
        BobReceives(b, bit1, bit2);

        let result = M(b);
        if result == Zero { set zeros += 1; }
        else { set ones += 1; }

        ResetAll([msg, a, b]);
    }

    return (zeros, ones);
}

In [ ]:
print("Teleporting |0> 10000 times:", qsharp.eval("TeleportMany(false, 10000)"))
print("Teleporting |1> 10000 times:", qsharp.eval("TeleportMany(true, 10000)"))
print("\nPerfect teleportation! The state is always correctly transferred.")

---
## 3. Superdense Coding in Q#

The dual protocol: send 2 classical bits using 1 qubit.

In [ ]:
%%qsharp

// Alice encodes a 2-bit message onto her qubit
operation AliceEncodes(bit1 : Bool, bit2 : Bool, aliceQubit : Qubit) : Unit {
    if bit2 { X(aliceQubit); }
    if bit1 { Z(aliceQubit); }
}

// Bob decodes the message
operation BobDecodes(aliceQubit : Qubit, bobQubit : Qubit) : (Result, Result) {
    CNOT(aliceQubit, bobQubit);
    H(aliceQubit);
    return (M(aliceQubit), M(bobQubit));
}

// Full superdense coding protocol
operation SuperdenseCoding(bit1 : Bool, bit2 : Bool) : (Result, Result) {
    use (aliceQubit, bobQubit) = (Qubit(), Qubit());

    // Create shared Bell pair
    H(aliceQubit);
    CNOT(aliceQubit, bobQubit);

    // Alice encodes her 2-bit message
    AliceEncodes(bit1, bit2, aliceQubit);

    // Bob decodes
    let (r1, r2) = BobDecodes(aliceQubit, bobQubit);

    ResetAll([aliceQubit, bobQubit]);
    return (r1, r2);
}

In [ ]:
# Test all four messages
messages = [(False, False), (False, True), (True, False), (True, True)]

for b1, b2 in messages:
    result = qsharp.eval(f"SuperdenseCoding({str(b1).lower()}, {str(b2).lower()})")
    expected = f"{int(b1)}{int(b2)}"
    print(f"Sent: {expected} -> Received: {result}")

print("\nAll messages decoded correctly!")

---
## 4. Key Takeaways

| Concept | Description |
|---------|-------------|
| **No-Cloning** | Cannot copy unknown quantum states |
| **Teleportation** | Transfer state using entanglement + 2 classical bits |
| **Superdense Coding** | Send 2 classical bits using 1 entangled qubit |
| **Q# Modularity** | Clean separation into Alice/Bob operations |

---
## 5. Challenges

1. **Teleport superposition**: Modify `Teleport` to teleport the |+> state. Run 10,000 times — do you get 50/50?
2. **Verify all superdense messages**: Run each superdense coding message 1000 times to confirm 100% accuracy.
3. **Teleport with Ry**: Teleport a state created by `Ry(theta, q)` for various angles. Verify Bob gets the same distribution.

In [ ]:
%%qsharp

// Your challenge code here!

---
**Next up: [Level 5 — Oracles and the Deutsch-Jozsa Algorithm](../Level_05_Oracles_Deutsch_Jozsa/Level_05_QSharp.ipynb)**